## Imports

In [1]:
import wandb
import logging
from tqdm import tqdm
from wandb.sdk.wandb_run import Run
import numpy as np
import pandas as pd
import plotly.express as px
import numpy as np
import plotly.graph_objs as go
import seaborn as sns
import matplotlib.pyplot as plt
from nn_core.common import PROJECT_ROOT
import json

/media/donato/Extra-storage/Code/model-merging/mass/.venv/lib/python3.11/site-packages/lightning_utilities/core/imports.py:14: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/media/donato/Extra-storage/Code/model-merging/mass/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/media/donato/Extra-storage/Code/model-merging/mass/.venv/lib/python3.11/site-packages/transformers/utils/hub.py:124: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


## Configuration

In [2]:
from mass.utils.plots import Palette

plt.rcParams.update(
    {
        "text.usetex": True,
        "font.family": "serif",
        "axes.titlesize": 24,        # Larger axes/title fonts
        "axes.labelsize": 24,
        "xtick.labelsize": 24,
        "ytick.labelsize": 20,
        "legend.fontsize": 24,
    }
)
sns.set_context("talk")

cmap_name = "coolwarm_r"

palette = Palette(f"{PROJECT_ROOT}/misc/palette.json", map_path=f"{PROJECT_ROOT}/misc/palette_map.json")
palette

Project not installed in the current env, activate the correct env or install it with:
	pip install -e .


{'blue': '#335c67',
 'white': '#fff3b0',
 'yellow': '#e09f3e',
 'red': '#9e2a2b',
 'dark red': '#540b0e',
 'green': '#81b29a'}

## Get runs

In [3]:
api = wandb.Api()
entity, project = "gladia", "mass"  # set to your entity and project

In [4]:
def get_runs(entity, project, positive_tags, negative_tags):
    filters_pos_tags = {"$and": [{"tags": {"$eq": pos_tag}} for pos_tag in positive_tags]}
    filters_neg_tags = {}

    print(filters_pos_tags)
    filters = {**filters_pos_tags, **filters_neg_tags}
    runs = api.runs(entity + "/" + project, filters=filters)

    print(f"There are {len(runs)} runs respecting these conditions.")
    return runs

In [5]:
tags = [
    "finetune"
]  

In [6]:
runs = get_runs(entity, project, positive_tags=tags, negative_tags=[])

{'$and': [{'tags': {'$eq': 'finetune'}}]}
There are 20 runs respecting these conditions.


In [17]:
print(set(runs[0].history().columns))

{'acc/test/DTD', 'acc/train/DTD', 'epoch', '_step', 'loss/train/DTD', '_timestamp', 'loss/test/DTD', 'trainer/global_step', '_runtime'}


In [8]:
models = ['ViT-B-32'] # ['ViT-B-32', 'ViT-B-16', 'ViT-L-14']
datasets =  ['Cars', 'DTD', 'EuroSAT', 'GTSRB', 'MNIST', 'RESISC45', 'SUN397', 'SVHN', 'CIFAR100', 'STL10', 'Flowers102', 'OxfordIIITPet', 'PCAM', 'FER2013', 'EMNIST', 'CIFAR10', 'Food101', 'FashionMNIST', 'RenderedSST2', 'KMNIST']

In [24]:
accs = {model: {dataset: None for dataset in datasets} for model in models}

#### Hparams

In [31]:
run_by_model = {}

for model in models:

    for run in runs:

            cols = run.history().columns
            print(cols)
            # Quirk: config wasn't uploaded when finetuned 
            for dataset in datasets:
                  for col in cols:
                      col_dataset = col.split('/')[-1]
                      if dataset == col_dataset:
                          print(f'Found dataset {dataset}')
                          accs[model][dataset] = run.history()[f'acc/test/{dataset}'].values[-1]
                          break # Found the dataset, break

Index(['loss/test/DTD', '_runtime', '_timestamp', 'acc/test/DTD', '_step',
       'epoch', 'loss/train/DTD', 'trainer/global_step', 'acc/train/DTD'],
      dtype='object')
Found dataset DTD
Index(['acc/train/MNIST', 'epoch', 'acc/test/MNIST', 'loss/test/MNIST',
       '_step', '_runtime', 'loss/train/MNIST', 'trainer/global_step',
       '_timestamp'],
      dtype='object')
Found dataset MNIST
Index(['_timestamp', 'acc/test/SVHN', 'trainer/global_step', 'loss/test/SVHN',
       'loss/train/SVHN', 'epoch', '_runtime', 'acc/train/SVHN', '_step'],
      dtype='object')
Found dataset SVHN
Index(['loss/test/GTSRB', '_step', '_timestamp', 'acc/train/GTSRB',
       'loss/train/GTSRB', 'trainer/global_step', '_runtime', 'acc/test/GTSRB',
       'epoch'],
      dtype='object')
Found dataset GTSRB
Index(['acc/test/EuroSAT', 'trainer/global_step', '_step', 'epoch',
       'loss/test/EuroSAT', '_runtime', '_timestamp', 'loss/train/EuroSAT',
       'acc/train/EuroSAT'],
      dtype='object')
Found 

In [32]:
accs

{'ViT-B-32': {'Cars': 0.7981594204902649,
  'DTD': 0.7819148898124695,
  'EuroSAT': 0.9925925731658936,
  'GTSRB': 0.989311158657074,
  'MNIST': 0.9944999814033508,
  'RESISC45': 0.9530158638954163,
  'SUN397': 0.748765766620636,
  'SVHN': 0.9666564464569092,
  'CIFAR100': 0.8695999979972839,
  'STL10': 0.9754999876022339,
  'Flowers102': 0.898682713508606,
  'OxfordIIITPet': 0.9269555807113647,
  'PCAM': 0.86260986328125,
  'FER2013': 0.7128726840019226,
  'EMNIST': 0.9944999814033508,
  'CIFAR10': 0.9753000140190125,
  'Food101': 0.8603564500808716,
  'FashionMNIST': 0.9416000247001648,
  'RenderedSST2': 0.6787479519844055,
  'KMNIST': 0.9812999963760376}}